## 4. Estimation

Linear probability model of `patented` on network exposure. Firm × subclass + year FE, clustered by firm.
Three regressor forms: cumulative stock, log(1+stock), binary.

In [1]:
import numpy as np
import pandas as pd
import pyfixest as pf

data_dir = r"C:\Users\Popov\Documents\Studies\IMT_studies\Projects\Econ_2\Datasets"
panel = pd.read_parquet(data_dir + r"\panel_firm_subclass_year.parquet")

FIRM, PARENT, SUB, YEAR = "bvd_key", "parent_key", "subclass", "year"
Y, X = "patented", "net_stock"

print(f"firms {panel[FIRM].nunique():,} | groups {panel[PARENT].nunique():,} | "
      f"subclasses {panel[SUB].nunique():,} | years {panel[YEAR].min()}-{panel[YEAR].max()}")
print(f"outcome mean {panel[Y].mean():.4f} | exposed cells {(panel[X] > 0).mean():.3f}")
print(f"own_subclass cells {panel['own_subclass'].mean():.3f} | "
      f"parent_is_foreign missing {panel['parent_is_foreign'].isna().sum():,}")

firms 674 | groups 264 | subclasses 527 | years 2015-2024
outcome mean 0.0362 | exposed cells 0.602
own_subclass cells 0.381 | parent_is_foreign missing 0


### Variables

Panel is firm × IPC subclass × year, 94,010 cells, 2015–2024. One row = one technology
class a firm could have patented in that year.

| Variable | Role | Definition |
|---|---|---|
| `bvd_key` | id | Italian affiliate (Orbis BvD ID, asterisk stripped) |
| `parent_key` | id | Global ultimate owner of the group (Rungi's GUO algorithm) |
| `subclass` | id | IPC subclass, 4 characters (e.g. C07D) |
| `year` | id | Priority year |
| `patented` | **y** | 1 if the firm filed a new patent family in this subclass this year |
| `n_new` | — | Count of new families behind `patented`; descriptive only |
| `net_stock` | **x** | Cumulative families in this subclass filed *before* t by the firm's Italian co-affiliates, excluding families the firm co-owns |
| `log_net` | x | log(1 + `net_stock`) |
| `net_any` | x | 1 if `net_stock` > 0 |
| `parent_is_foreign` | split | 1 if the GUO is non-Italian |
| `firm_year`, `sub_year` | FE keys | Pre-built interactions for alternative FE structures |
| `own_subclass` | flag | Marks the firm's own technology space — defined on in-window patenting, so not usable as a sample restriction |

**Sample.** Patenting affiliates whose group contains at least one *other* patenting
affiliate (674 firms); firms outside that have `net_stock` = 0 everywhere.

**Timing.** Priority date throughout (grant date as robustness). `net_stock` is a strict
lag — only families filed before t enter — so there is no mechanical same-year overlap
between x and y.

**Self-exclusion.** Families the firm itself co-owns are removed from `net_stock`, so the
regressor is never the firm's own patenting re-labelled.

### Regressor forms and the identifying sample

With firm × subclass FE the estimate comes from *within-pair* movement. A pair where the
regressor never changes across the window contributes nothing to it.

In [2]:
panel["pair"] = panel[FIRM] + "|" + panel[SUB]

var = panel.groupby("pair").agg(y_var=(Y, "nunique"),
                                x_var=(X, "nunique"),
                                any_var=("net_any", "nunique"))

print(f"pairs: {len(var):,}")
print(f"  y varies within pair:         {(var.y_var > 1).sum():,}")
print(f"  net_stock varies within pair: {(var.x_var > 1).sum():,}")
print(f"  net_any varies within pair:   {(var.any_var > 1).sum():,}")

both = var.index[(var.y_var > 1) & (var.x_var > 1)]
print(f"  both move: {len(both):,} pairs | {panel['pair'].isin(both).sum():,} cells | "
      f"{panel.loc[panel['pair'].isin(both), FIRM].nunique():,} firms")

pairs: 9,401
  y varies within pair:         1,676
  net_stock varies within pair: 3,069
  net_any varies within pair:   1,744
  both move: 293 pairs | 2,930 cells | 151 firms


#### Identifying sample per regressor

Each regressor is scoped separately — the foreign/domestic split below is computed inside
the loop, so each line reports its own regressor's identifying firms.

In [3]:
for v in [X, "log_net", "net_any"]:
    vv = panel.groupby("pair").agg(y_var=(Y, "nunique"), x_var=(v, "nunique"))
    moves = vv.index[vv.x_var > 1]
    idp = vv.index[(vv.x_var > 1) & (vv.y_var > 1)]
    idf = panel.loc[panel["pair"].isin(idp), [FIRM, "parent_is_foreign"]].drop_duplicates()
    split = idf["parent_is_foreign"].value_counts(dropna=False).to_dict()
    print(f"{v:10s} | x moves {len(moves):5,} pairs | y also moves {len(idp):4,} | "
          f"{idf[FIRM].nunique():3,} firms | split {split}")

net_stock  | x moves 3,069 pairs | y also moves  293 | 151 firms | split {0.0: 107, 1.0: 44}
log_net    | x moves 3,069 pairs | y also moves  293 | 151 firms | split {0.0: 107, 1.0: 44}
net_any    | x moves 1,744 pairs | y also moves  118 |  77 firms | split {0.0: 51, 1.0: 26}


With firm × subclass FE, a pair only identifies the coefficient if the regressor moves
within it. `net_stock` moves in 3,069 of 9,401 pairs, but `patented` also moves in only
293 of those — 151 firms. The remaining 2,776 pairs add variance without signal, so the
baseline is attenuated by construction.

`log_net` is monotone in `net_stock` and identifies off the same 293 pairs. `net_any`
identifies off 118 pairs (77 firms): it asks a narrower question — what happens when a
network enters a technology from scratch mid-window — and is not the same effect in
binary form.

Identifying firms are 29% foreign-parent against 21% in the full sample, so foreign
affiliates are over-represented among identifiers. Cluster counts for the split are 107
domestic / 44 foreign.

### Baseline

In [4]:
fe = f"pair + {YEAR}"
cl = {"CRV1": FIRM}

m_stock = pf.feols(f"{Y} ~ {X}      | {fe}", data=panel, vcov=cl)
m_log   = pf.feols(f"{Y} ~ log_net  | {fe}", data=panel, vcov=cl)
m_any   = pf.feols(f"{Y} ~ net_any  | {fe}", data=panel, vcov=cl)

pf.etable([m_stock, m_log, m_any])

GT(_tbl_data=  __index_level_0__ __index_level_1__                    0  \
0              coef         net_stock  -0.000 <br> (0.000)   
1              coef           log_net                        
2              coef           net_any                        
3                fe              pair                    -   
4                fe              year                    -   
5             stats      Observations               94,010   
6             stats                R²                0.346   

                    1                   2  
0                                          
1  0.005 <br> (0.004)                      
2                      0.005 <br> (0.003)  
3                   -                   -  
4                   -                   -  
5              94,010              94,010  
6               0.346               0.346  , _body=<great_tables._gt_data.Body object at 0x0000021540830260>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000002154095D3D0>, _spanners=Spanners([SpannerInfo(spanner_id='patented', spanner_level=1, spanner_label=Html(text='patented'), spanner_units=None, spanner_pattern=None, vars=['0', '1', '2'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000002154095C4A0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000002154095D0D0>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000002154095C950>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000002154088F2F0>, <great_tables._gt_data.FormatInfo object at 0x000002154095CB60>, <great_tables._gt_data.FormatInfo object at 0x000002154095C6E0>, <great_tables._gt_data.FormatInfo object at 0x000002154095CF50>, <great_tables._gt_data.FormatInfo object at 0x000002154095C7A0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category

etable rounds to three decimals, so the levels coefficient prints as 0.000 and its magnitude is invisible. The cell below prints the same estimates at full precision — read the levels column there, not in the table above.

In [5]:
for m in [m_stock, m_log, m_any]:
    print(m.tidy().to_string(float_format=lambda v: f"{v:.3e}"), "\n")

              Estimate  Std. Error    t value  Pr(>|t|)       2.5%     97.5%
Coefficient                                                                 
net_stock   -1.991e-04   1.607e-04 -1.238e+00 2.160e-01 -5.146e-04 1.165e-04 

             Estimate  Std. Error   t value  Pr(>|t|)       2.5%     97.5%
Coefficient                                                               
log_net     4.964e-03   3.503e-03 1.417e+00 1.570e-01 -1.915e-03 1.184e-02 

             Estimate  Std. Error   t value  Pr(>|t|)       2.5%     97.5%
Coefficient                                                               
net_any     4.749e-03   3.479e-03 1.365e+00 1.727e-01 -2.082e-03 1.158e-02 



In [6]:
print(pf.etable([m_stock, m_log, m_any], type="tex"))

\begin{threeparttable}
\begingroup
\renewcommand\cellalign{t}
\renewcommand\arraystretch{1}
\setlength{\tabcolsep}{3pt}
\begin{tabularx}{\linewidth}{@{}>{\raggedright\arraybackslash}l>{\centering\arraybackslash}X>{\centering\arraybackslash}X>{\centering\arraybackslash}X}
\toprule
 & \multicolumn{3}{c}{patented} \\
\cmidrule(lr){2-4}
 & (1) & (2) & (3) \\
\midrule
\addlinespace[1ex]
net_stock & \makecell{-0.000 \\ (0.000)} &  &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
log_net &  & \makecell{0.005 \\ (0.004)} &  \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
net_any &  &  & \makecell{0.005 \\ (0.003)} \\
\addlinespace[0.5ex]
\midrule
\addlinespace[1ex]
pair & - & - & - \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
year & - & - & - \\
\addlinespace[0.5ex]
\midrule
\addlinespace[1ex]
Observations & 94,010 & 94,010 & 94,010 \\
\addlinespace[0.5ex]
\addlinespace[0.5ex]
$R^2$ & 0.346 & 0.346 & 0.346 \\
\addlinespace[0.5ex]
\bottomrule
\end{tabularx}
\endgroup
\noindent\begin{minipage}{\line

Insignificant results. So the question is whether this is an absence of spillovers or an
absence of power. 293 identifying pairs against 3,069 carrying variance — the diagnostic
below separates the two.

### Movers only

Restricting to the 3,069 pairs where the regressor moves drops pairs that contribute
zero to the estimator by construction — not a selection on the outcome. Point estimates
should be close to the baseline with tighter SEs. If a coefficient moves materially
instead, the full-grid estimate was being held down by something other than dead weight.

Note what this filter is **not**: conditioning on `patented` varying within the pair would
select cells because the outcome turned on, which inflates the coefficient roughly sixfold against the full-grid baseline, and twelvefold against the movers estimate below.
The filter is on x only.

In [7]:
idp = panel.groupby("pair")[X].nunique() > 1
mv = panel[panel["pair"].isin(idp.index[idp])].copy()
mv["firm_id"] = mv[FIRM].astype("category").cat.codes   # numeric cluster id

print(f"{len(mv):,} cells | {mv['pair'].nunique():,} pairs | {mv['firm_id'].nunique():,} firms")

b_stock = pf.feols(f"{Y} ~ {X}      | {fe}", data=mv, vcov=cl)
b_log   = pf.feols(f"{Y} ~ log_net  | {fe}", data=mv, vcov=cl)
b_any   = pf.feols(f"{Y} ~ net_any  | {fe}", data=mv, vcov=cl)

pf.etable([b_stock, b_log, b_any])

30,690 cells | 3,069 pairs | 505 firms


GT(_tbl_data=  __index_level_0__ __index_level_1__                      0  \
0              coef         net_stock  -0.000** <br> (0.000)   
1              coef           log_net                          
2              coef           net_any                          
3                fe              pair                      -   
4                fe              year                      -   
5             stats      Observations                 30,690   
6             stats                R²                  0.399   

                    1                   2  
0                                          
1  0.002 <br> (0.003)                      
2                      0.002 <br> (0.004)  
3                   -                   -  
4                   -                   -  
5              30,690              30,690  
6               0.399               0.399  , _body=<great_tables._gt_data.Body object at 0x00000214DB7E8DA0>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000021541121B80>, _spanners=Spanners([SpannerInfo(spanner_id='patented', spanner_level=1, spanner_label=Html(text='patented'), spanner_units=None, spanner_pattern=None, vars=['0', '1', '2'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x0000021541120050>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x00000215411205C0>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x0000021541120710>, _formats=[<great_tables._gt_data.FormatInfo object at 0x0000021541120290>, <great_tables._gt_data.FormatInfo object at 0x0000021541120980>, <great_tables._gt_data.FormatInfo object at 0x0000021541120950>, <great_tables._gt_data.FormatInfo object at 0x0000021541120800>, <great_tables._gt_data.FormatInfo object at 0x0000021541120140>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scs

In [8]:
for m in [b_stock, b_log, b_any]:
    print(m.tidy().to_string(float_format=lambda v: f"{v:.3e}"), "\n")

              Estimate  Std. Error    t value  Pr(>|t|)       2.5%      97.5%
Coefficient                                                                  
net_stock   -3.307e-04   1.275e-04 -2.594e+00 9.757e-03 -5.811e-04 -8.025e-05 

             Estimate  Std. Error   t value  Pr(>|t|)       2.5%     97.5%
Coefficient                                                               
log_net     2.423e-03   3.227e-03 7.511e-01 4.530e-01 -3.916e-03 8.763e-03 

             Estimate  Std. Error   t value  Pr(>|t|)       2.5%     97.5%
Coefficient                                                               
net_any     1.793e-03   3.610e-03 4.966e-01 6.197e-01 -5.300e-03 8.886e-03 



Movers-only halves log_net (0.0050 → 0.0024) and cuts net_any to under 40% of its baseline, so the flat-regressor pairs were not purely inert — they were contributing some positive covariance, not only denominator.

Read the levels column in the full-precision output, not the table. On the full grid
`net_stock` is indistinguishable from zero, but on the movers sample it is significantly
**negative** at conventional levels. The magnitude is on the order of 1e-4 per additional
network family, and it runs opposite to both other forms — which suggests it is picking up
the near-monotone trend in a cumulative stock rather than a spillover. Reported, not
interpreted.

### Inference

**Why the asymptotic SE may fail here.** The CRV1 variance estimator is

$$\hat{V} = c\,(X'X)^{-1}\Big(\sum_{g=1}^{G} X_g'\hat{u}_g\hat{u}_g'X_g\Big)(X'X)^{-1},
\qquad c=\frac{G}{G-1}\cdot\frac{N-1}{N-K}$$

The middle term averages over $G$, not $N$, so consistency needs $G\to\infty$. Residuals
are shrunk by the projection, so $\hat{V}$ is biased down and $t$ does not converge to
$N(0,1)$ — nominal 5% tests can over-reject.

**What $G$ is here.** Clustering on `bvd_key` gives $G = 674$ firms on the full grid, 505
on the movers sample (264 groups if clustered on `parent_key` instead). But a firm whose
pairs all have a flat regressor has demeaned $x_g = 0$, so it contributes a zero block to
$\sum_g X_g'\hat u_g\hat u_g'X_g$ while still counting in the correction $c$. The meat
matrix is effectively summed over the **151 firms** that identify the coefficient. That
gap — large nominal $G$, small effective $G$ — is the reason to check inference rather
than take the default SEs on trust.

**Wild cluster bootstrap: not feasible here.** The natural check would impose the null,
resample cluster-wise and bootstrap the $t$-statistic rather than the coefficient. The
`wildboottest` implementation one-hot encodes the fixed effects, so the movers sample needs
a 30,690 × 3,079 dense design matrix and several copies of it; the full grid needs
94,010 × 9,411. Neither fits in memory on this machine. (Cluster ids must also be numeric —
`bvd_key` is a string and numba rejects it, hence the `.cat.codes` conversion above, kept
in case the bootstrap is run elsewhere.)

**What is done instead.** Compare three variance estimators on the same specification:
firm-level CRV1, heteroskedasticity-robust, and parent-group clustering. Parent level is
the substantively motivated one — co-affiliates' exposure is built from each other's
patents, so sister firms are not independent draws, and the coarser level is arguably
correct. Agreement across the three is the evidence that inference is not fragile.

It is also dropped for comparability: the risk-set design in notebook 5 carries three sets of fixed effects and is further out of reach, so bootstrapping only this specification would leave the two notebooks checking inference by different standards.

References: Cameron, Gelbach & Miller (2008); Roodman, Nielsen, MacKinnon & Webb (2019);
MacKinnon, Nielsen & Webb (2023) for the survey.

In [9]:
inf4 = []
for v in [X, "log_net", "net_any"]:
    inf4.append(pf.feols(f"{Y} ~ {v} | {fe}", data=mv, vcov={"CRV1": "firm_id"}))
    inf4.append(pf.feols(f"{Y} ~ {v} | {fe}", data=mv, vcov="hetero"))
    inf4.append(pf.feols(f"{Y} ~ {v} | {fe}", data=mv, vcov={"CRV1": PARENT}))

pf.etable(inf4)

GT(_tbl_data=  __index_level_0__ __index_level_1__                      0  \
0              coef         net_stock  -0.000** <br> (0.000)   
1              coef           log_net                          
2              coef           net_any                          
3                fe              pair                      -   
4                fe              year                      -   
5             stats      Observations                 30,690   
6             stats                R²                  0.399   

                     1                      2                   3  \
0  -0.000 <br> (0.000)  -0.000** <br> (0.000)                       
1                                              0.002 <br> (0.003)   
2                                                                   
3                    -                      -                   -   
4                    -                      -                   -   
5               30,690                 30,690              30,690   
6                0.399                  0.399               0.399   

                    4                   5                   6  \
0                                                               
1  0.002 <br> (0.002)  0.002 <br> (0.004)                       
2                                          0.002 <br> (0.004)   
3                   -                   -                   -   
4                   -                   -                   -   
5              30,690              30,690              30,690   
6               0.399               0.399               0.399   

                    7                   8  
0                                          
1                                          
2  0.002 <br> (0.003)  0.002 <br> (0.004)  
3                   -                   -  
4                   -                   -  
5              30,690              30,690  
6               0.399               0.399  , _body=<great_tables._gt_data.Body object at 0x0000021540C961B0>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(4)'), column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(5)'), column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(6)'), column_align='center', column_width=None), ColInfo(var='6', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(7)'), column_align='center', column_width=None), ColInfo(var='7', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(8)'), column_align='center', column_width=None), ColInfo(var='8', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(9)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000021540C957F0>, _spanners=Spanners([SpannerInfo(spanner_id='patented', spanner_level=1, spanner_label=Html(text='patented'), spanner_units=None, spanner_pattern=None, vars=['0', '1', '2', '3', '4', '5', '6', '7', '8'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x0000021540C95DC0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x0000021540C95A90>, _source_notes

In [10]:
it4 = pf.etable(inf4, type="df")
it4 = it4.drop("fe", level=0)

it4.loc[("spec", "Standard errors"), :]  = ["Firm", "Robust", "Parent group"] * 3
it4.loc[("fe", "Firm-subclass FE"), :] = ["Yes"] * 9
it4.loc[("fe", "Year FE"), :]          = ["Yes"] * 9

it4 = it4.reindex(["coef", "spec", "fe", "stats"], level=0)
it4 = it4.replace({r"\n": " "}, regex=True)
it4

patented                    \
                                       (1)               (2)   
coef  net_stock         -0.000**   (0.000)  -0.000   (0.000)   
      log_net                                                  
      net_any                                                  
spec  Standard errors                 Firm            Robust   
fe    Firm-subclass FE                 Yes               Yes   
      Year FE                          Yes               Yes   
stats Observations                  30,690            30,690   
      R²                             0.399             0.399   

                                                                              \
                                       (3)              (4)              (5)   
coef  net_stock         -0.000**   (0.000)                                     
      log_net                               0.002   (0.003)  0.002   (0.002)   
      net_any                                                                  
spec  Standard errors         Parent group             Firm           Robust   
fe    Firm-subclass FE                 Yes              Yes              Yes   
      Year FE                          Yes              Yes              Yes   
stats Observations                  30,690           30,690           30,690   
      R²                             0.399            0.399            0.399   

                                                                           \
                                    (6)              (7)              (8)   
coef  net_stock                                                             
      log_net           0.002   (0.004)                                     
      net_any                            0.002   (0.004)  0.002   (0.003)   
spec  Standard errors      Parent group             Firm           Robust   
fe    Firm-subclass FE              Yes              Yes              Yes   
      Year FE                       Yes              Yes              Yes   
stats Observations               30,690           30,690           30,690   
      R²                          0.399            0.399            0.399   

                                         
                                    (9)  
coef  net_stock                          
      log_net                            
      net_any           0.002   (0.004)  
spec  Standard errors      Parent group  
fe    Firm-subclass FE              Yes  
      Year FE                       Yes  
stats Observations               30,690  
      R²                          0.399

In [11]:
iout = it4.copy()
iout.index = iout.index.droplevel(0)
iout = iout.rename(index={"net_stock": "Network stock",
                          "log_net": "log(1 + stock)",
                          "net_any": "Network present",
                          "R²": "$R^2$"})

print(iout.to_latex(escape=False, column_format="l" + "c" * 9,
                    caption="Movers sample under alternative standard errors",
                    label="tab:inference_nb4"))

\begin{table}
\caption{Movers sample under alternative standard errors}
\label{tab:inference_nb4}
\begin{tabular}{lccccccccc}
\toprule
 & \multicolumn{9}{r}{patented} \\
 & (1) & (2) & (3) & (4) & (5) & (6) & (7) & (8) & (9) \\
\midrule
Network stock & -0.000**   (0.000) & -0.000   (0.000) & -0.000**   (0.000) &  &  &  &  &  &  \\
log(1 + stock) &  &  &  & 0.002   (0.003) & 0.002   (0.002) & 0.002   (0.004) &  &  &  \\
Network present &  &  &  &  &  &  & 0.002   (0.004) & 0.002   (0.003) & 0.002   (0.004) \\
Standard errors & Firm & Robust & Parent group & Firm & Robust & Parent group & Firm & Robust & Parent group \\
Firm-subclass FE & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes \\
Year FE & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes \\
Observations & 30,690 & 30,690 & 30,690 & 30,690 & 30,690 & 30,690 & 30,690 & 30,690 & 30,690 \\
$R^2$ & 0.399 & 0.399 & 0.399 & 0.399 & 0.399 & 0.399 & 0.399 & 0.399 & 0.399 \\
\bottomrule
\end{tabular}
\end{table}



Inference is stable for the two interpretable forms. `log_net` is 0.002 under all three
estimators, with SEs of 0.003 (firm), 0.002 (robust) and 0.004 (parent group); `net_any`
behaves the same. Parent-group clustering widens the interval relative to firm level, as
expected when co-affiliates share the regressor by construction, but not enough to change
any conclusion. The notebook 4 question about cluster level is settled: firm level is
adequate, and the null holds at the coarser level regardless.

The levels form is the exception. `net_stock` is significant at 1% under both firm and
parent clustering but not under heteroskedasticity-robust SEs — its significance depends
on the estimator, while the other two forms do not. Note the direction: the clustered SEs are smaller than the robust one, which happens only when within-cluster residual correlation is negative — clustering normally widens them. That is a further sign the levels coefficient reflects mechanical structure in a cumulative stock rather than an effect.

### Summary

Baseline LPM: `log_net` 0.005 (0.004) and `net_any` 0.005 (0.003), neither distinguishable
from zero; levels indistinguishable from zero on the full grid.

Movers-only roughly halves both, and the levels form turns small but significantly negative
(order 1e-4 per family) — reported, not interpreted, since it runs against both other forms
and most likely tracks the monotone trend in a cumulative stock.

Cause is thin identification: under firm × subclass FE, only 293 pairs across 151 firms
have both the regressor and the outcome moving.

Failure to reject, not evidence of no effect — the interval covers zero and an effect twice
the descriptive cross-tab (13.0% vs 8.6%). Notebook 5 addresses the power problem with a
pre-determined risk set and a firm × year / subclass × year design.